# Session 10 — Your turn: Evaluating LLMs

Two arithmetic habits for reading an evaluation: put an interval on every accuracy, and check whether a judge is reading the answers or the positions.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from aij_check import check, close, summary

## Task 1: An interval on an accuracy

A model answered 41 of 60 questions correctly. Write `wilson(k, n, z=1.96)` returning the Wilson 95% interval `(low, high)` for the true accuracy:
with `p = k / n`, centre `(p + z**2 / (2n)) / (1 + z**2 / n)` and half-width `z * sqrt(p(1-p)/n + z**2 / (4n**2)) / (1 + z**2 / n)`. Store the interval for 41/60 in `interval` and for 410/600 in `interval_big`, and `narrower_by`, how many times narrower the second one is.

In [2]:
def wilson(k, n, z=1.96):
    p = k / n
    centre = (p + z ** 2 / (2 * n)) / (1 + z ** 2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / (1 + z ** 2 / n)
    return centre - half, centre + half


interval = wilson(41, 60)
interval_big = wilson(410, 600)
narrower_by = (interval[1] - interval[0]) / (interval_big[1] - interval_big[0])
print(interval, interval_big, narrower_by)

(np.float64(0.5576599505698345), np.float64(0.7869429415882567)) (np.float64(0.6450453939726326), np.float64(0.7192885638070237)) 3.0882704972035464


In [3]:
check("the interval for 41/60 is about 0.56 to 0.79", lambda: close(interval, [0.5600, 0.7863], 0.005))
check("it contains the observed accuracy", lambda: interval[0] < 41 / 60 < interval[1])
check("ten times the questions give an interval about 3 times narrower", lambda: 2.9 < narrower_by < 3.4)
check("it stays inside 0 and 1 even for a perfect score", lambda: 0 <= wilson(20, 20)[0] and wilson(20, 20)[1] <= 1.0 + 1e-9)
summary()

PASS  the interval for 41/60 is about 0.56 to 0.79
PASS  it contains the observed accuracy
PASS  ten times the questions give an interval about 3 times narrower
PASS  it stays inside 0 and 1 even for a perfect score

4 of 4 checks passed.  Well done.


**Why it works.** With 60 questions a score of 68% could be anywhere from 56% to 79%, so two models 5 points apart cannot be ranked. The width shrinks with the square root of the number of questions: ten times more questions, three times narrower.

## Task 2: Is the judge reading the answers or the positions?

A language-model judge compared 200 pairs of answers twice, once with answer A first and once with the order swapped. `first_pass` and `second_pass` hold which **position** it picked (`"first"` or `"second"`) in each pass.
Compute `first_share` (the share of picks of the first position over both passes), `consistent` (the share of pairs where it picked the **same answer** in both passes; picking the first position in pass one and the second position in pass two is consistent) and `reads_positions` (True if `first_share` is above 0.7).

In [4]:
rng = np.random.default_rng(0)
quality = rng.random(200) < 0.5
noise = rng.random(200)
first_pass = np.where(noise < 0.7, "first", np.where(quality, "first", "second"))
second_pass = np.where(noise < 0.7, "first", np.where(quality, "second", "first"))
first_share = float(np.mean(np.concatenate([first_pass, second_pass]) == "first"))
consistent = float(np.mean(first_pass != second_pass))
reads_positions = bool(first_share > 0.7)
print(first_share, consistent, reads_positions)

0.825 0.35 True


In [5]:
check("the judge picks the first position about 85% of the time", lambda: close(first_share, 0.85, 0.05))
check("it picks the same answer in both orders only about 30% of the time", lambda: close(consistent, 0.30, 0.07), "Picking 'first' in both passes means two different answers were chosen.")
check("it is reading positions", lambda: reads_positions is True)
summary()

PASS  the judge picks the first position about 85% of the time
PASS  it picks the same answer in both orders only about 30% of the time
PASS  it is reading positions

3 of 3 checks passed.  Well done.


**Why it works.** A judge that reads the answers gives the same answer in both orders. This one prefers the first slot 85% of the time and agrees with itself only on pairs where the preference for the first slot did not decide. The session finds a 0.5B judge picking the first answer 97.9% of the time; running both orders and counting inconsistencies is the cheap test.